## 1. Initialize the project

Create the working context: data download project if not created already. Project is a placeholder for the code, data, and management of the data operations.

In [ ]:
import digitalhub as dh
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## 2. Execution

Fetch the "vllm-kubeai-serve" operation in the project. 

In [ ]:
llm_vllm_func = proj.get_function('vllm-kubeai-serve')

Alternatively, one can create new function

In [ ]:
llm_vllm_func = proj.new_function(
    name="vllm-kubeai-serve",
    kind="kubeai-text",
    model_name="llm-vllm-model",
    url="hf://Qwen/Qwen2.5-0.5B-Instruct",
    engine='VLLM',
    features=['TextGeneration']
)

Run the function with default configuration, 

In [ ]:
vllm_run = llm_vllm_func.run("serve", profile="1xV100", args=["--dtype=float16"], wait=True)

Wait for job to finish. The state of 'serve' run can be viewed running the cell below.



In [ ]:
status = vllm_run.refresh().status.k8s.get("Model")['status']
print("Model status:", status)

Note: Alternatively, using the Core Management UI, one can navigate to 'Runs' menu , select the corresponding 'run' instance and inspect the logs using 'Logs' tab.

To fetch the deployed service URL and model ID, you can access the relevant fields from the `vllm_run` object after the function has been run. The service URL is available under the `service` key, and the model ID can be found under the `openai` key in the status dictionary. Use the following code to retrieve and print these values:

- `CHAT_URL` will contain the service URL.
- `CHAT_MODEL` will contain the model ID.

These variables can then be used for making API requests or further testing.

In [ ]:
service = vllm_run.refresh().status.service
print("Service status:", service)

In [ ]:
CHAT_URL = vllm_run.status.to_dict()["service"]["url"]
CHAT_MODEL = vllm_run.status.to_dict()["openai"]["model"]
print(f"service {CHAT_URL} with model {CHAT_MODEL}")

## 3. Test the API

To invoke the deployed API, you need to provide the model identifier and a prompt. This is done by constructing a JSON payload with the required fields and sending it to the service endpoint.

**Steps:**
1. Retrieve the model name and prepare the JSON payload:
    - `model`: The model ID obtained from the deployment.
    - `prompt`: The text prompt you want the model to complete or respond to.

    ```python
    model_name = llm_run.refresh().status.k8s.get("Model").get("metadata").get("name")
    json_payload = {'model': CHAT_MODEL, 'prompt': 'Describe MLOps'}
    ```

2. Invoke the API using the service URL and payload:
    - Use the `.invoke()` method with the `json` argument and the appropriate endpoint.

    ```python
    result = llm_run.invoke(json=json_payload, url=service['url']+'/v1/completions').json()
    ```

3. Process and display the response as needed.



In [ ]:
model_name =vllm_run.refresh().status.k8s.get("Model").get("metadata").get("name")
json_payload = {'model': CHAT_MODEL, 'prompt': 'Describe MLOPs'}
json_payload

In [ ]:
import pprint
pp = pprint.PrettyPrinter(indent=2)
result = vllm_run.invoke(json=json_payload, url=service['url']+'/v1/completions').json()
print("Response:")
pp.pprint(result)

In [ ]:
Response:
{ 'choices': [ { 'finish_reason': 'length',
                 'index': 0,
                 'logprobs': None,
                 'prompt_logprobs': None,
                 'prompt_token_ids': None,
                 'stop_reason': None,
                 'text': ' and how they work in the context of a specific ML '
                         'model.\n'
                         'MLOps',
                 'token_ids': None}],
  'created': 1772012102,
  'id': 'cmpl-a589857d516958f0',
  'kv_transfer_params': None,
  'model': 'model-17499be858474e9580e4ea819730b000',
  'object': 'text_completion',
  'service_tier': None,
  'system_fingerprint': None,
  'usage': { 'completion_tokens': 16,
             'prompt_tokens': 4,
             'prompt_tokens_details': None,
             'total_tokens': 20}}